# Model Comparison

## Overview

In the previous notebook, we established a baseline by training and evaluating a Linear Regression model using a train-validation split. While this provided an initial understanding of the model's predictive performance, the evaluation was based on a single validation split, which may not provide a reliable estimate of the model's generalization performance.

In this notebook, we compare Linear Regression and three regularized regression models using **5-fold Cross-Validation**. Unlike a single train-validation split, cross-validation evaluates each model across multiple validation folds, reducing the dependence on any particular data split and providing a more robust estimate of model performance.

The models compared in this notebook are:

- Linear Regression
- Ridge Regression (L2 Regularization)
- Lasso Regression (L1 Regularization)
- ElasticNet Regression (L1 + L2 Regularization)

Each model is evaluated using the same preprocessing pipeline, cross-validation strategy, and evaluation metrics to ensure a fair comparison.

---

## Objectives

The objectives of this notebook are to:

- Evaluate multiple linear regression models using 5-fold Cross-Validation.
- Compare model performance using consistent evaluation metrics.
- Analyze the impact of different regularization techniques on model generalization.
- Identify the most promising model(s) for hyperparameter tuning.
---

## Notebook Workflow

```text
Initialize Model
        │
        ▼
Perform 5-Fold Cross-Validation
        │
        ▼
Compute Mean Performance
        │
        ▼
Interpret Results
        │
        ▼
Compare Models
        │
        ▼
Select Model(s) for Hyperparameter Tuning
```

---

## Evaluation Metrics

Each model is evaluated using the following metrics:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Coefficient of Determination (R²)

For each metric, the **mean** and **standard deviation** across the five cross-validation folds are reported. The mean provides an estimate of the model's average performance, while the standard deviation indicates the consistency of that performance across different validation folds.

---

## Expected Outcome

By the end of this notebook, we will:

- Compare the cross-validation performance of Linear Regression, Ridge Regression, Lasso Regression, and ElasticNet Regression.
- Assess the stability and generalization performance of each model.
- Select the most suitable model(s) for hyperparameter tuning in the next stage of the project.

---

In [1]:
#import libraries

import numpy as np
import pandas as pd



#set parh for access src module
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

#import resusable utility 
from src.pipeline import create_pipeline
from src.evaluation import evaluate_cv_model


#pandas display setting 
pd.options.display.float_format = "{:.4f}".format

In [2]:
# Load training dataset
train_df = pd.read_csv("../data/train.csv")

# Separate features and target
X_train = train_df.drop(columns=["SalePrice"])
y_train = train_df["SalePrice"]

##  Cross-Validation Strategy

Before comparing the regression models, a **5-fold Cross-Validation** strategy is adopted to obtain a more reliable estimate of each model's generalization performance.

In a single train-validation split, the evaluation metrics may vary depending on how the dataset is partitioned. As a result, the observed performance may not accurately represent the model's ability to generalize to unseen data.

To reduce this dependence on a single split, **5-fold Cross-Validation** divides the training dataset into five approximately equal-sized folds. During each iteration, four folds are used to train the model, while the remaining fold is used for validation. This process is repeated five times, allowing every observation to serve as the validation set exactly once.

The evaluation metrics obtained from the five validation folds are then averaged to provide a more robust estimate of model performance.

For this project:

- **5-fold Cross-Validation** is used.
- The same cross-validation strategy is applied to every model.
- The mean and standard deviation of each evaluation metric are reported for comparison.

---

## Cross-Validation Configuration

A **5-fold K-Fold Cross-Validation** strategy is used to evaluate all regression models in this notebook.

The configuration is defined as follows:

- **Number of folds (`n_splits=5`)**: Divides the training dataset into five approximately equal-sized folds. During each iteration, four folds are used for training and one fold is used for validation.
- **Shuffling (`shuffle=True`)**: Randomly shuffles the observations before creating the folds, reducing the possibility of biased data partitions.
- **Random state (`random_state=42`)**: Ensures that the same folds are generated each time the notebook is executed, making the evaluation reproducible.

In addition, a common set of evaluation metrics is specified and reused across all models. This ensures that every regression model is assessed under identical evaluation criteria, enabling a fair comparison of their predictive performance.

In [3]:
from sklearn.model_selection import KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

----

## 1. Baseline Model-Linear Regression

### 1.1 Model Overview

Linear Regression serves as the baseline model for this project. It models the relationship between the input features and the target variable by fitting a linear equation that minimizes the residual sum of squares between the observed and predicted values.

Unlike regularized regression models, Linear Regression does not apply any penalty to the model coefficients. As a result, it attempts to fit the training data solely by minimizing prediction error, which may increase the risk of overfitting when the dataset contains correlated features or noisy observations.

The purpose of evaluating this model is to establish a reference point against which the performance of regularized regression models can be compared.


---

### 1.2 Model Initialization

The first model evaluated is the baseline **Linear Regression** model. Unlike the previous notebook, where the model was evaluated using a single train-validation split, this notebook evaluates the model using **5-fold Cross-Validation**.

The same preprocessing pipeline developed earlier in the project is reused to ensure consistent data preparation across all models. The Linear Regression estimator is appended as the final step of the pipeline, allowing every cross-validation fold to independently fit both the preprocessing transformations and the regression model without introducing data leakage.

In [4]:
from sklearn.linear_model import LinearRegression
linear_pipeline = create_pipeline(LinearRegression())


----

### 1.3 Cross-Validation Evaluation

The Linear Regression pipeline is evaluated using the predefined 5-fold Cross-Validation strategy.

For each fold, the preprocessing pipeline is fitted using only the corresponding training subset before generating predictions on the validation subset. The evaluation metrics obtained from all five validation folds are then aggregated to estimate the model's overall predictive performance and consistency.

In [5]:
linear_cv_results = evaluate_cv_model(
    pipeline=linear_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
)

linear_cv_results

,Train Mean,Train Std,Validation Mean,Validation Std
MAE,12564.2796,527.8750,20170.5572,2557.7352
RMSE,19050.0992,1401.0257,45731.7599,19310.9724
R²,0.9448,0.0067,0.6204,0.3113


----

### 1.4 Interpretation

The Linear Regression model achieves strong performance on the training folds, with an average **R² score of 0.9448**, indicating that it explains approximately **94.48%** of the variance in the training data. The relatively low training MAE and RMSE further suggest that the model fits the training data effectively.

However, the model's performance declines on the validation folds. The average validation **R² score decreases to 0.6204**, while both the MAE and RMSE increase considerably compared with their training counterparts. This noticeable gap between the training and validation metrics indicates that the model does not generalize well to unseen data.

The standard deviation of the validation **R² score (0.3113)** and **RMSE (19310.9724)** is also relatively high, suggesting that the model's performance varies substantially across different validation folds. This indicates that the model is sensitive to the particular training-validation split used during cross-validation.

Overall, the Linear Regression model exhibits **overfitting (high variance)**. Although it learns the training data effectively, its ability to generalize to unseen data is limited. Consequently, it serves as a useful baseline for evaluating whether regularized regression models such as Ridge, Lasso, and ElasticNet can improve generalization performance.


---

##  1.  Ridge Regression

### 2.1 Model Overview

Ridge Regression extends Linear Regression by incorporating **L2 regularization**, which adds a penalty proportional to the squared magnitude of the model coefficients during training.

The regularization term discourages excessively large coefficient values, resulting in a simpler and more stable model. This helps reduce model complexity and can improve generalization, particularly when the dataset contains multicollinearity or when the baseline Linear Regression model exhibits signs of overfitting.

---

### 2.2 Model Initialization

The same preprocessing pipeline is reused to ensure consistent data preparation and a fair comparison with the baseline model. The Ridge Regression estimator is added as the final step of the pipeline using its default hyperparameters.

As with the baseline model, the preprocessing transformations and regression model are fitted independently within each cross-validation iteration, ensuring that no information from the validation fold is used during training.

In [7]:
from sklearn.linear_model import Ridge

ridge_pipeline = create_pipeline(Ridge())

---

### 2.3 Cross-Validation Evaluation

The Ridge Regression pipeline is evaluated using the predefined 5-fold cross-validation strategy.

During each iteration, the model is trained on four folds and evaluated on the remaining validation fold. This process is repeated five times, allowing every observation to serve as the validation set exactly once. The evaluation metrics are then aggregated to assess the model's average predictive performance and stability across different data splits.

In [8]:
ridge_cv_results = evaluate_cv_model(
    pipeline=ridge_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
)

ridge_cv_results

,Train Mean,Train Std,Validation Mean,Validation Std
MAE,13933.0312,800.2052,19485.0125,2518.7902
RMSE,21058.2719,1580.2219,34511.6850,11992.3238
R²,0.9325,0.0083,0.8043,0.1307


---

### 2.4 Interpretation

The Ridge Regression model achieves strong performance on the training folds, with an average **R² score of 0.9325**, indicating that it explains approximately **93.25%** of the variance in the training data. Although the training performance is slightly lower than that of the baseline Linear Regression model, this reduction is expected because L2 regularization constrains the magnitude of the model coefficients.

More importantly, the model demonstrates a substantial improvement on the validation folds. The average validation **R² score increases to 0.8043**, while both the validation MAE and RMSE decrease compared with the baseline model. This indicates that Ridge Regression generalizes significantly better to unseen data.

The gap between the training and validation metrics is noticeably smaller than that observed for the baseline Linear Regression model, suggesting that the introduction of L2 regularization has successfully reduced overfitting. Furthermore, the relatively lower standard deviation of the validation metrics, particularly the **R² standard deviation of 0.1307**, indicates that the model performs more consistently across different validation folds.

Overall, Ridge Regression provides a better balance between model fit and generalization than the baseline Linear Regression model. These results demonstrate that L2 regularization effectively reduces model variance while improving predictive performance on unseen data, making Ridge Regression a strong candidate for further hyperparameter tuning.

----

## 2. Lasso Regression

### 3.1 Model Overview

Lasso Regression extends Linear Regression by introducing **L1 regularization**, which adds a penalty proportional to the absolute magnitude of the model coefficients during training.

Unlike Ridge Regression, Lasso Regression can shrink some coefficients exactly to zero, thereby performing automatic feature selection while fitting the model. This property can improve model interpretability and may reduce the influence of less informative features.


---

### 3.2 Model Initialization

The same preprocessing pipeline is reused to ensure consistent data preparation and a fair comparison across all regression models. The Lasso Regression estimator is added as the final step of the pipeline using its default hyperparameters.

During each cross-validation iteration, both the preprocessing transformations and the regression model are fitted exclusively on the training folds before being evaluated on the corresponding validation fold. This prevents data leakage and ensures an unbiased estimate of model performance.

In [10]:
from sklearn.linear_model import Lasso

lasso_pipeline = create_pipeline(
    Lasso(max_iter=50000)
)


----

### 3.3 Cross-Validation Evaluation

The Lasso Regression pipeline is evaluated using the predefined 5-fold cross-validation strategy.

During each iteration, the model is trained on four folds and evaluated on the remaining validation fold. This process is repeated five times, allowing every observation to serve as the validation set exactly once. The evaluation metrics obtained from all folds are then aggregated to estimate the model's average predictive performance and consistency across different data splits.

In [11]:
lasso_cv_results = evaluate_cv_model(
    pipeline=lasso_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
)

lasso_cv_results

,Train Mean,Train Std,Validation Mean,Validation Std
MAE,12581.3228,530.2560,19571.3010,2773.6166
RMSE,19058.3462,1402.0551,43166.1075,17558.0970
R²,0.9448,0.0067,0.6685,0.2494


----

### 3.4 Interpretation

The Lasso Regression model achieves strong performance on the training folds, with an average **R² score of 0.9448**, indicating that it explains approximately **94.48%** of the variance in the training data. The training performance is nearly identical to that of the baseline Linear Regression model, suggesting that the default L1 regularization strength has only a limited effect on the fitted model.

On the validation folds, the model achieves an average **R² score of 0.6685**, representing an improvement over the baseline Linear Regression model. The validation MAE and RMSE also decrease, indicating better predictive performance on unseen data. However, the improvement is less pronounced than that achieved by Ridge Regression.

Although Lasso Regression reduces the gap between the training and validation metrics compared with the baseline model, a noticeable difference still exists, indicating that some degree of overfitting remains. In addition, the validation **R² standard deviation of 0.2494** suggests moderate variability in performance across different validation folds, making the model less stable than Ridge Regression.

Overall, Lasso Regression improves generalization compared with the baseline Linear Regression model but does not outperform Ridge Regression when using the default hyperparameters. Further hyperparameter tuning may improve its ability to balance feature selection and predictive performance.

----

## 3. ElasticNet Regression

### 4.1 Model Overview

ElasticNet Regression combines **L1** and **L2 regularization**, integrating the feature selection capability of Lasso Regression with the coefficient shrinkage property of Ridge Regression.

By balancing these two regularization techniques, ElasticNet can reduce model complexity while mitigating some of the limitations of using L1 or L2 regularization individually. It is particularly useful when the dataset contains highly correlated features, where Lasso may select one feature arbitrarily and Ridge may retain all correlated features.


---

### 4.2 Model Initialization

The same preprocessing pipeline is reused to ensure consistent data preparation and a fair comparison across all regression models. The ElasticNet estimator is added as the final step of the pipeline using its default hyperparameters.

During each cross-validation iteration, the preprocessing transformations and the regression model are fitted exclusively on the training folds before being evaluated on the corresponding validation fold. This ensures that the validation data remains unseen during training and prevents data leakage.

In [12]:
from sklearn.linear_model import ElasticNet

elasticnet_pipeline = create_pipeline(
    ElasticNet()
)

----

### 4.3 Cross-Validation Evaluation

The ElasticNet pipeline is evaluated using the predefined 5-fold cross-validation strategy.

During each iteration, the model is trained on four folds and evaluated on the remaining validation fold. This process is repeated five times, allowing every observation to be used once for validation. The evaluation metrics obtained from all folds are then aggregated to estimate the model's average predictive performance and consistency across different data splits.

In [13]:
elasticnet_cv_results = evaluate_cv_model(
    pipeline=elasticnet_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
)

elasticnet_cv_results

,Train Mean,Train Std,Validation Mean,Validation Std
MAE,18925.3729,570.6718,20005.2455,1922.4867
RMSE,33338.5006,2268.0380,36623.2146,9750.7084
R²,0.8310,0.0191,0.7881,0.1029


---

### 4.4 Interpretation

The ElasticNet Regression model achieves an average **training R² score of 0.8310**, which is lower than those of the other regression models evaluated. This reduction in training performance is expected, as the combined L1 and L2 regularization imposes stronger constraints on the model coefficients, resulting in a simpler model.

Despite the lower training performance, ElasticNet demonstrates strong generalization on the validation folds, achieving an average **R² score of 0.7881**. The validation RMSE is substantially lower than that of the baseline Linear Regression and Lasso Regression models, although it remains slightly higher than that of Ridge Regression.

The relatively small gap between the training and validation metrics suggests that ElasticNet effectively reduces overfitting by controlling model complexity. Furthermore, the validation **R² standard deviation of 0.1029**, the lowest among all evaluated models, indicates that ElasticNet produces the most consistent performance across different validation folds.

Overall, ElasticNet provides a good balance between model simplicity and predictive performance. While its average validation performance is slightly lower than Ridge Regression, its greater stability across folds suggests that it generalizes reliably. Consequently, ElasticNet represents a strong candidate for further hyperparameter tuning alongside Ridge Regression.

----

##  Model Comparison

In [14]:
comparison_table = pd.DataFrame({
    "Train R²": [
        linear_cv_results.loc["R²", "Train Mean"],
        ridge_cv_results.loc["R²", "Train Mean"],
        lasso_cv_results.loc["R²", "Train Mean"],
        elasticnet_cv_results.loc["R²", "Train Mean"],
    ],
    "Validation R²": [
        linear_cv_results.loc["R²", "Validation Mean"],
        ridge_cv_results.loc["R²", "Validation Mean"],
        lasso_cv_results.loc["R²", "Validation Mean"],
        elasticnet_cv_results.loc["R²", "Validation Mean"],
    ],
    "Validation RMSE": [
        linear_cv_results.loc["RMSE", "Validation Mean"],
        ridge_cv_results.loc["RMSE", "Validation Mean"],
        lasso_cv_results.loc["RMSE", "Validation Mean"],
        elasticnet_cv_results.loc["RMSE", "Validation Mean"],
    ],
    "Validation MAE": [
        linear_cv_results.loc["MAE", "Validation Mean"],
        ridge_cv_results.loc["MAE", "Validation Mean"],
        lasso_cv_results.loc["MAE", "Validation Mean"],
        elasticnet_cv_results.loc["MAE", "Validation Mean"],
    ],
    "Validation R² Std": [
        linear_cv_results.loc["R²", "Validation Std"],
        ridge_cv_results.loc["R²", "Validation Std"],
        lasso_cv_results.loc["R²", "Validation Std"],
        elasticnet_cv_results.loc["R²", "Validation Std"],
    ],
}, index=[
    "Linear Regression",
    "Ridge Regression",
    "Lasso Regression",
    "ElasticNet Regression",
])

comparison_table.round(4)

,Train R²,Validation R²,Validation RMSE,Validation MAE,Validation R² Std
Linear Regression,0.9448,0.6204,45731.7599,20170.5572,0.3113
Ridge Regression,0.9325,0.8043,34511.6850,19485.0125,0.1307
Lasso Regression,0.9448,0.6685,43166.1075,19571.3010,0.2494
ElasticNet Regression,0.8310,0.7881,36623.2146,20005.2455,0.1029


## Findings

The cross-validation results indicate clear differences in the predictive performance and generalization ability of the evaluated regression models.

The baseline **Linear Regression** model achieved excellent performance on the training folds but exhibited a noticeable decline in validation performance, indicating that it was prone to overfitting. This established an appropriate baseline against which the regularized regression models could be compared.

Among the regularized models, **Ridge Regression** achieved the strongest overall validation performance. It obtained the highest average validation **R² score** while simultaneously producing the lowest validation **MAE** and **RMSE**. In addition, the smaller gap between its training and validation metrics suggests that L2 regularization effectively reduced overfitting and improved the model's ability to generalize to unseen data.

**Lasso Regression** improved upon the baseline Linear Regression model but did not achieve the same level of predictive performance as Ridge Regression. Although L1 regularization helped improve generalization, its default configuration was less effective for this dataset than L2 regularization.

**ElasticNet Regression** demonstrated competitive validation performance while exhibiting the lowest variability across the validation folds. This indicates that its predictive performance was the most consistent across different data splits. However, its average validation performance remained slightly below that of Ridge Regression.

Overall, Ridge Regression provides the best balance between predictive accuracy and generalization among the evaluated models.

---

## Selected Models for Hyperparameter Tuning

Based on the cross-validation results, the following models are selected for hyperparameter tuning:

### Ridge Regression

Ridge Regression is selected as the **primary candidate** because it demonstrated the strongest overall predictive performance.

The model was selected for the following reasons:

- Achieved the highest average validation **R² score**.
- Produced the lowest validation **MAE** and **RMSE**.
- Reduced the gap between the training and validation metrics, indicating improved generalization.
- Maintained consistent performance across the cross-validation folds.

These results suggest that optimizing the regularization strength (`alpha`) may further improve its predictive performance.

### ElasticNet Regression

ElasticNet Regression is selected as a **secondary candidate** for hyperparameter tuning.

Although its average validation performance was slightly lower than Ridge Regression, it demonstrated the lowest variability across the validation folds, indicating highly stable and consistent performance. Since ElasticNet combines both L1 and L2 regularization, tuning its `alpha` and `l1_ratio` parameters may further improve its predictive performance.

---

## Conclusion

In this notebook, four linear regression models were evaluated using a consistent **5-fold cross-validation** strategy. The comparison demonstrated that regularization substantially improved model generalization compared with the baseline Linear Regression model.

Among the evaluated models, **Ridge Regression** achieved the best overall predictive performance, while **ElasticNet Regression** demonstrated the most consistent performance across different validation folds.

The next stage of the project focuses on **hyperparameter tuning**, where the selected models will be systematically optimized to identify the configuration that provides the best balance between predictive accuracy and generalization on unseen data.

----